In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

In [3]:
df  = pd.read_csv("books_data.csv")
df

,Title,description,authors,image,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount
0,Its Only Art If Its Well Hung!,NaN,['Julie Strain'],http://books.google.com/books/content?id=DykPA...,http://books.google.nl/books?id=DykPAAAACAAJ&d...,NaN,1996,http://books.google.nl/books?id=DykPAAAACAAJ&d...,['Comics & Graphic Novels'],NaN
1,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,['Philip Nel'],http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...,A&C Black,2005-01-01,http://books.google.nl/books?id=IjvHQsCn_pgC&d...,['Biography & Autobiography'],NaN
2,Wonderful Worship in Smaller Churches,This resource includes twelve principles in un...,['David R. Ray'],http://books.google.com/books/content?id=2tsDA...,http://books.google.nl/books?id=2tsDAAAACAAJ&d...,NaN,2000,http://books.google.nl/books?id=2tsDAAAACAAJ&d...,['Religion'],NaN
3,Whispers of the Wicked Saints,Julia Thomas finds her life spinning out of co...,['Veronica Haddon'],http://books.google.com/books/content?id=aRSIg...,http://books.google.nl/books?id=aRSIgJlq6JwC&d...,iUniverse,2005-02,http://books.google.nl/books?id=aRSIgJlq6JwC&d...,['Fiction'],NaN
4,"Nation Dance: Religion, Identity and Cultural ...",NaN,['Edward Long'],NaN,http://books.google.nl/books?id=399SPgAACAAJ&d...,NaN,2003-03-01,http://books.google.nl/books?id=399SPgAACAAJ&d...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
212399,The Orphan Of Ellis Island (Time Travel Advent...,"During a school trip to Ellis Island, Dominick...",['Elvira Woodruff'],http://books.google.com/books/content?id=J7M-N...,http://books.google.com/books?id=J7M-NwAACAAJ&...,Scholastic Paperbacks,2000-06-01,http://books.google.com/books?id=J7M-NwAACAAJ&...,['Juvenile Fiction'],2.0
212400,Red Boots for Christmas,Everyone in the village of Friedensdorf is hap...,NaN,http://books.google.com/books/content?id=3n8k6...,http://books.google.com/books?id=3n8k6wl4BbYC&...,NaN,1995,http://books.google.com/books?id=3n8k6wl4BbYC&...,['Juvenile Fiction'],NaN
212401,Mamaw,"Give your Mamaw a useful, beautiful and though...",['Wild Wild Cabbage'],NaN,http://books.google.com/books?id=zytVswEACAAJ&...,NaN,2018-01-17,http://books.google.com/books?id=zytVswEACAAJ&...,NaN,NaN
212402,The Autograph Man,Alex-Li Tandem sells autographs. His business ...,['Zadie Smith'],http://books.google.com/books/content?id=JM6YV...,http://books.google.com/books?id=JM6YVPx_clMC&...,Vintage,2003-08-12,https://play.google.com/store/books/details?id...,['Fiction'],19.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 212404 entries, 0 to 212403
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Title          212403 non-null  object 
 1   description    143962 non-null  object 
 2   authors        180991 non-null  object 
 3   image          160329 non-null  object 
 4   previewLink    188568 non-null  object 
 5   publisher      136518 non-null  object 
 6   publishedDate  187099 non-null  object 
 7   infoLink       188568 non-null  object 
 8   categories     171205 non-null  object 
 9   ratingsCount   49752 non-null   float64
dtypes: float64(1), object(9)
memory usage: 16.2+ MB


In [5]:
clean_df = df[["Title", "description", "authors", "categories", "image", "previewLink" ]]
clean_df

,Title,description,authors,categories,image,previewLink
0,Its Only Art If Its Well Hung!,NaN,['Julie Strain'],['Comics & Graphic Novels'],http://books.google.com/books/content?id=DykPA...,http://books.google.nl/books?id=DykPAAAACAAJ&d...
1,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,['Philip Nel'],['Biography & Autobiography'],http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...
2,Wonderful Worship in Smaller Churches,This resource includes twelve principles in un...,['David R. Ray'],['Religion'],http://books.google.com/books/content?id=2tsDA...,http://books.google.nl/books?id=2tsDAAAACAAJ&d...
3,Whispers of the Wicked Saints,Julia Thomas finds her life spinning out of co...,['Veronica Haddon'],['Fiction'],http://books.google.com/books/content?id=aRSIg...,http://books.google.nl/books?id=aRSIgJlq6JwC&d...
4,"Nation Dance: Religion, Identity and Cultural ...",NaN,['Edward Long'],NaN,NaN,http://books.google.nl/books?id=399SPgAACAAJ&d...
...,...,...,...,...,...,...
212399,The Orphan Of Ellis Island (Time Travel Advent...,"During a school trip to Ellis Island, Dominick...",['Elvira Woodruff'],['Juvenile Fiction'],http://books.google.com/books/content?id=J7M-N...,http://books.google.com/books?id=J7M-NwAACAAJ&...
212400,Red Boots for Christmas,Everyone in the village of Friedensdorf is hap...,NaN,['Juvenile Fiction'],http://books.google.com/books/content?id=3n8k6...,http://books.google.com/books?id=3n8k6wl4BbYC&...
212401,Mamaw,"Give your Mamaw a useful, beautiful and though...",['Wild Wild Cabbage'],NaN,NaN,http://books.google.com/books?id=zytVswEACAAJ&...
212402,The Autograph Man,Alex-Li Tandem sells autographs. His business ...,['Zadie Smith'],['Fiction'],http://books.google.com/books/content?id=JM6YV...,http://books.google.com/books?id=JM6YVPx_clMC&...


In [6]:
clean_data = clean_df.copy()
clean_data.dropna(subset=["Title", "description", "authors", "categories", "image", "previewLink" ], inplace=True)

In [22]:
clean_data

,Title,description,authors,categories,image,previewLink
1,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,Philip Nel,Biography & Autobiography,http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...
2,Wonderful Worship in Smaller Churches,This resource includes twelve principles in un...,David R. Ray,Religion,http://books.google.com/books/content?id=2tsDA...,http://books.google.nl/books?id=2tsDAAAACAAJ&d...
3,Whispers of the Wicked Saints,Julia Thomas finds her life spinning out of co...,Veronica Haddon,Fiction,http://books.google.com/books/content?id=aRSIg...,http://books.google.nl/books?id=aRSIgJlq6JwC&d...
5,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,Everett Ferguson,Religion,http://books.google.com/books/content?id=kVqRa...,http://books.google.nl/books?id=kVqRaiPlx88C&p...
8,Saint Hyacinth of Poland,The story for children 10 and up of St. Hyacin...,Mary Fabyan Windeatt,Biography & Autobiography,http://books.google.com/books/content?id=lmLqA...,http://books.google.nl/books?id=lmLqAAAACAAJ&d...
...,...,...,...,...,...,...
212394,Final things,Grace's father believes in science and builds ...,Jenny Offill,Fiction,http://books.google.com/books/content?id=UbSFB...,http://books.google.com/books?id=UbSFBAAAQBAJ&...
212397,The Magic of the Soul: Applying Spiritual Powe...,"""The Magic of the Soul, Applying Spiritual Pow...",Patrick J. Harbula,"Body, Mind & Spirit",http://books.google.com/books/content?id=H1ELA...,http://books.google.com/books?id=H1ELAAAACAAJ&...
212398,Autodesk Inventor 10 Essentials Plus,Autodesk Inventor 2017 Essentials Plus provide...,"Daniel Banach, Travis Jones",Computers,http://books.google.com/books/content?id=zxHRC...,http://books.google.com/books?id=zxHRCwAAQBAJ&...
212399,The Orphan Of Ellis Island (Time Travel Advent...,"During a school trip to Ellis Island, Dominick...",Elvira Woodruff,Juvenile Fiction,http://books.google.com/books/content?id=J7M-N...,http://books.google.com/books?id=J7M-NwAACAAJ&...


In [8]:
clean_data.isnull().sum()

Title          0
description    0
authors        0
categories     0
image          0
previewLink    0
dtype: int64

In [11]:
clean_data.duplicated().sum()

np.int64(0)

In [12]:
clean_data.describe()

,Title,description,authors,categories,image,previewLink
count,130818,130818,130818,130818,130818,130818
unique,130818,121440,94043,4456,122825,130425
top,The Autograph Man,Publisher Description,['Agatha Christie'],['Fiction'],http://books.google.com/books/content?id=dPucx...,http://books.google.com/books?id=acwPAgAAQBAJ&...
freq,1,54,105,21842,28,17


In [15]:
useless_data = clean_data[clean_data["description"].apply(lambda x: len(str(x).split()) < 10)]
useless_data

,Title,description,authors,categories,image,previewLink
132,Sheikh (Mills & Boon Historical),"""I'd rather eat nails than bend to his will!""",['Sharon De Vita'],['Fiction'],http://books.google.com/books/content?id=TrbvA...,http://books.google.nl/books?id=TrbvAgAAQBAJ&p...
235,"The Jester's Quest (Winds of Light, Book 7)",No description available.,['Ann Ann Howey'],['History'],http://books.google.com/books/content?id=3C_hG...,http://books.google.nl/books?id=3C_hGEGdqcIC&p...
263,Social Research Methods: Qualitative and Quant...,Includes bibliographical references and index.,['William Lawrence Neuman'],['Social Science'],http://books.google.com/books/content?id=rKL9l...,http://books.google.nl/books?id=rKL9lgWN_1gC&q...
312,Environmental Science: Toward A Sustainable Fu...,Resource added for the Solar Energy Technology...,"['Richard T. Wright', 'Bernard J. Nebel']",['Science'],http://books.google.com/books/content?id=wXjA3...,http://books.google.nl/books?id=wXjA3AUHEIYC&q...
404,Body Self: take ACTION in your quest for Posit...,(see back cover image that I sent.),['Melissa Dodd'],['Self-Help'],http://books.google.com/books/content?id=hXg4b...,http://books.google.nl/books?id=hXg4beXsUm8C&p...
...,...,...,...,...,...,...
211952,Pedra Canga (Green Integer: 32),A major new novel by Brazilian writer Tereza A...,['Tereza Albues'],['Fiction'],http://books.google.com/books/content?id=MGotA...,http://books.google.com/books?id=MGotAAAAYAAJ&...
211994,Night Shift (Signet),A chilling collection of twenty horror stories.,['Stephen King'],"['Horror tales, American']",http://books.google.com/books/content?id=p9TRi...,http://books.google.com/books?id=p9TRiBVo7TcC&...
211999,Internet Kids & Family Yellow Pages,Provides strategies for keeping children and t...,['Nancy E. Willard'],['Computers'],http://books.google.com/books/content?id=fDo7f...,http://books.google.com/books?id=fDo7f-ldz_EC&...
212047,The Connoisseurs Book of Japanese Swords,Connoisseur's Book Japanese Swords is a Kodans...,['Kōkan Nagayama'],['Antiques & Collectibles'],http://books.google.com/books/content?id=zPysw...,http://books.google.com/books?id=zPyswmGDBFkC&...


In [16]:
clean_data.drop(
    clean_data[clean_data["description"].apply(lambda x: len(str(x).split()) < 10)].index,
    inplace=True
)

In [21]:
clean_data["categories"] = clean_data["categories"].str.strip("[]").str.replace("'", "")
clean_data["authors"] = clean_data["authors"].str.strip("[]").str.replace("'", "")

In [23]:
clean_data.to_csv("filtered_data.csv", index=False)

In [27]:
df = pd.read_csv("filtered_data.csv")
df["unique_values"] = range(1, len(df) + 1)

df["tagged_description"] = df[["unique_values", "description"]].astype(str).agg(" ".join, axis=1)
df.to_csv("complete_data.csv", index=False)

print("Unique values added successfully!")


Unique values added successfully!
